# Mars-calibration regime pipeline (regA, regB, or regC)

End-to-end retraining of Earth XGBoost + CNN under a single pruning regime,
followed by Mars inference. Run cell-by-cell; interrupt and edit
`SKIP_BASINS` when you need to drop a slow basin (e.g. Taiwan at regA).

Regimes (defined in `scripts/build_earth_features_regime.py`):

| name | T (km²) | pre_remove ≤ | order_gap ≥ | character                       |
|------|--------:|-------------:|------------:|---------------------------------|
| regA | 0.05    | 2            | 4           | most aggressive prune, densest start |
| regB | 0.25    | 1            | 4           | least aggressive, sparsest start     |
| regC | 0.10    | 1            | 4           | middle ground (T between A and B)    |

All regimes use:
- the per-outlet prefilter (`2 × sqrt(basin_area_pixels)` px, floored at
  `regime.min_prefilter_px = 30`), so prefilter distance scales with actual
  drainage area instead of a fixed magic number;
- parallel coupling (`evaluate_pairs_for_outlet_parallel(n_workers=4)`).

Output artifacts use `_regA` / `_regB` / `_regC` suffixes; production models
and `master_dataset_v2.csv` are never touched.

**Steps:** 2 features → 3 patches → 4 CNN train → 5 combined XGB → 6 Mars inference.


## 0. Configuration

Change `REGIME` and `SKIP_BASINS` here, then run cells in order.

In [1]:
# ─── Pick regime ──────────────────────────────────────────────────────
REGIME = "regC"   # "regA" or "regB"

# Basins to skip in Step 2 (and downstream). Useful if regA is too slow on
# big DEMs like Taiwan / Daqing / Sakhalin.
SKIP_BASINS: list[str] = []  # e.g. ["taiwan", "daqing"]

# Per-basin runtime bounds (mirror earth_network_pruning notebook defaults).
MIN_BASIN_PX = 500
MAX_OUTLETS = 0

# Mirror nb00 negative subsampling.
NEGATIVE_RATIO = 3.0
RANDOM_SEED = 42
HARD_NEG_MAX_L_RATIO = 3.0
HARD_NEG_MAX_DIST_RATIO = 5.0

# If True, skip per-basin work when full_features_<regime>.csv already exists.
USE_CACHED_BASIN_CSV = True

In [2]:
# ─── Imports & repo root resolution ───────────────────────────────────
import os, sys, time, gc, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import topotoolbox as tt3
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("CHANNEL_HEADS_ROOT", str(PROJECT_ROOT))
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from channel_heads import (
    CouplingAnalyzer,
    GeometricFeaturesAnalyzer,
    LengthwiseAsymmetryAnalyzer,
    apply_strategy,
    filter_hard_negatives,
    first_meet_pairs_for_outlet,
    generate_labeled_dataset,
    outlet_node_ids_from_streampoi,
)
from channel_heads.basin_config import LOCAL_TO_PAPER_BASIN, get_basin_config
from channel_heads.io.paths import EXAMPLE_DEMS, RESULTS_DIR
from channel_heads.dd_calibration import (
    _node_rowcol,
    compute_pixel_size_m_from_dem,
    compute_threshold_cells,
    linear_index_fortran,
)

# Bring in the regime presets + stratified_subsample_negatives helper.
from build_earth_features_regime import (  # noqa: E402
    DEM_TO_BASIN,
    REGIMES,
    stratified_subsample_negatives,
)

regime = REGIMES[REGIME]
print(f"Project root : {PROJECT_ROOT}")
print(f"Regime       : {regime.name}  (T={regime.threshold_km2} km², "
      f"pre_remove≤{regime.pre_remove_max_order}, gap≥{regime.order_gap_to_prune})")
print(f"Skip basins  : {SKIP_BASINS or '(none)'}")

/Users/guypi/miniforge3/envs/ch-heads/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root : /Users/guypi/Projects/channel-heads
Regime       : regC  (T=0.1 km², pre_remove≤1, gap≥4)
Skip basins  : (none)


## Step 2 — Per-basin Earth features (regime-pruned)

Builds the regime-pruned `StreamObject` for each basin and runs the existing
coupling / asymmetry / geometric pipeline. Writes
`data/results/{basin}/full_features_<regime>.csv`.

**Interrupt-safe:** if a basin hangs, hit ⏹ Stop, add it to `SKIP_BASINS` in
the config cell, and re-run this loop. Cached basin CSVs are reused.

In [3]:
import math as _math

def process_basin_inline(basin_name: str, dem_path: Path) -> tuple[pd.DataFrame | None, dict]:
    """In-notebook version of scripts/build_earth_features_regime.py:process_basin.

    Uses CouplingAnalyzer.evaluate_pairs_for_outlet_parallel + a PER-OUTLET
    prefilter distance derived from each outlet's actual drainage area:
        prefilter_distance_px = 2 * sqrt(basin_area_pixels)
    Same formula as production, just with the outlet's real basin as input.
    """
    stats = {
        "basin": basin_name, "n_outlets": 0, "n_outlets_kept": 0,
        "n_outlets_with_pairs": 0, "n_pairs": 0, "n_touching": 0,
        "n_not_touching": 0, "threshold_cells": 0,
        "n_nodes_full": 0, "n_nodes_pruned": 0, "time_s": 0.0, "error": None,
    }
    t0 = time.time()
    cfg = get_basin_config(LOCAL_TO_PAPER_BASIN.get(basin_name, basin_name))
    z_th, lat = cfg["z_th"], float(cfg["lat"])

    try:
        dem = tt3.read_tif(str(dem_path))
        dem.z[dem.z < z_th] = np.nan
        pixel_size_m = compute_pixel_size_m_from_dem(dem, lat_deg=lat)
        cells = compute_threshold_cells(regime.threshold_km2, pixel_size_m)
        stats["threshold_cells"] = cells

        fd = tt3.FlowObject(dem)
        s_full = tt3.StreamObject(fd, threshold=cells)
        stats["n_nodes_full"] = int(np.asarray(s_full.node_indices[0]).size)

        s = apply_strategy(
            s_full,
            pre_remove_max_order=regime.pre_remove_max_order,
            order_gap_to_prune=regime.order_gap_to_prune,
        )
        if s is None:
            stats["error"] = "Pruning removed entire network"
            stats["time_s"] = time.time() - t0
            return None, stats
        stats["n_nodes_pruned"] = int(np.asarray(s.node_indices[0]).size)

        outlets = outlet_node_ids_from_streampoi(s)
        stats["n_outlets"] = int(len(outlets))
        if len(outlets) == 0:
            stats["error"] = "No outlets after pruning"
            stats["time_s"] = time.time() - t0
            return None, stats

        valid_dem = ~np.isnan(dem.z)
        n_rows = dem.z.shape[0]
        rows_all, cols_all = _node_rowcol(s)
        outlet_lin = np.array(
            [linear_index_fortran(rows_all[o], cols_all[o], n_rows) for o in outlets],
            dtype=np.int64,
        )
        labels = np.asarray(fd.drainagebasins(outlet_lin).z)
        valid_labels = labels[valid_dem].ravel().astype(np.int64)
        counts = np.bincount(valid_labels, minlength=outlets.size + 1)
        sizes = counts[1 : outlets.size + 1]
        keep_idx = np.flatnonzero(sizes >= MIN_BASIN_PX)
        if MAX_OUTLETS and keep_idx.size > MAX_OUTLETS:
            keep_idx = keep_idx[np.argsort(sizes[keep_idx])[::-1][:MAX_OUTLETS]]
        outlets_kept = outlets[keep_idx]
        sizes_kept = sizes[keep_idx]
        stats["n_outlets_kept"] = int(len(outlets_kept))
        if len(outlets_kept) == 0:
            stats["error"] = "No outlets pass size filter"
            stats["time_s"] = time.time() - t0
            return None, stats

        # CouplingAnalyzer init with placeholder threshold; we override
        # `_prefilter_distance` per OUTLET inside the loop using each
        # outlet`s actual drainage area in pixels.
        coupling_an = CouplingAnalyzer(
            fd, s, dem, connectivity=8, threshold=1
        )
        asym_an = LengthwiseAsymmetryAnalyzer(s, dem, lat=lat)
        geom_an = GeometricFeaturesAnalyzer(
            s, dem, lat=lat, node_orders=s.streamorder(method="strahler")
        )

        outlet_dfs: list[pd.DataFrame] = []
        for oid, basin_px in tqdm(
            list(zip(outlets_kept, sizes_kept)),
            desc=f"  {basin_name} outlets", leave=False,
        ):
            try:
                pairs, _ = first_meet_pairs_for_outlet(s, int(oid))
                if not pairs:
                    continue
                # Per-outlet prefilter distance from drainage area:
                # 2 * sqrt(basin_area_px) px (= production formula, real input).
                pf_px = max(
                    regime.min_prefilter_px,
                    2.0 * _math.sqrt(float(basin_px)),
                )
                coupling_an._prefilter_distance = pf_px

                coupling_df = coupling_an.evaluate_pairs_for_outlet_parallel(
                    int(oid), pairs, n_workers=regime.coupling_n_workers
                )
                if coupling_df.empty:
                    continue
                asym_df = asym_an.evaluate_pairs_for_outlet(int(oid), pairs)
                geom_df = geom_an.evaluate_pairs_for_outlet(
                    int(oid), pairs, asymmetry_df=asym_df
                )
                df_o = generate_labeled_dataset(coupling_df, asym_df, geom_df)
                if df_o is not None and not df_o.empty:
                    outlet_dfs.append(df_o)
                    stats["n_outlets_with_pairs"] += 1
            except Exception as exc:
                print(f"    [{basin_name}] outlet={int(oid)} failed: {exc}")
            finally:
                coupling_an.clear_cache()

        if not outlet_dfs:
            stats["error"] = "No pairs found in any outlet"
            stats["time_s"] = time.time() - t0
            return None, stats

        df_basin = pd.concat(outlet_dfs, ignore_index=True)
        df_basin["basin"] = basin_name
        stats["n_pairs"] = int(len(df_basin))
        stats["n_touching"] = int(df_basin["y"].sum())
        stats["n_not_touching"] = stats["n_pairs"] - stats["n_touching"]
        stats["time_s"] = time.time() - t0
        return df_basin, stats
    except Exception as exc:
        stats["error"] = str(exc)
        stats["time_s"] = time.time() - t0
        return None, stats


In [4]:
# ─── Per-basin loop ───────────────────────────────────────────────────
basin_records: list[pd.DataFrame] = []
basin_stats: list[dict] = []
skipped = set(b.lower() for b in SKIP_BASINS)

for dem_file, basin_name in sorted(DEM_TO_BASIN.items()):
    if basin_name in skipped:
        print(f"[{basin_name}] SKIPPED (in SKIP_BASINS)")
        continue
    dem_path = EXAMPLE_DEMS.get(basin_name)
    if dem_path is None or not Path(dem_path).exists():
        print(f"[{basin_name}] DEM missing on disk; skipping")
        continue

    cache_path = RESULTS_DIR / basin_name / f"full_features_{regime.name}.csv"
    if USE_CACHED_BASIN_CSV and cache_path.exists():
        df_basin = pd.read_csv(cache_path)
        print(f"[{basin_name}] cached: {len(df_basin)} pairs ({int((df_basin['y']==1).sum())} touching)")
        basin_records.append(df_basin)
        basin_stats.append({
            "basin": basin_name, "from_cache": True,
            "n_pairs": len(df_basin),
            "n_touching": int((df_basin["y"] == 1).sum()),
            "n_not_touching": int((df_basin["y"] == 0).sum()),
            "time_s": 0.0, "error": None,
        })
        continue

    print(f"\n[{basin_name}] processing ...")
    df_basin, stats = process_basin_inline(basin_name, dem_path)
    stats["from_cache"] = False
    basin_stats.append(stats)
    if df_basin is None or df_basin.empty:
        print(f"  ✗ FAILED in {stats.get('time_s', 0):.1f}s: {stats.get('error')}")
    else:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        df_basin.to_csv(cache_path, index=False)
        print(f"  ✓ {stats['n_pairs']} pairs ({stats['n_touching']} touching)"
              f" in {stats['time_s']:.1f}s → {cache_path}")
        basin_records.append(df_basin)
    gc.collect()

print("\nPer-basin loop done.")

[calnalpine] cached: 78 pairs (7 touching)
[daqing] cached: 92 pairs (8 touching)
[finisterre] cached: 15776 pairs (475 touching)
[humboldt] cached: 144 pairs (13 touching)
[inyo] cached: 178 pairs (26 touching)
[kammanasie] cached: 213 pairs (16 touching)
[luliang] cached: 544 pairs (25 touching)
[panamint] cached: 732 pairs (46 touching)
[sakhalin] cached: 444 pairs (30 touching)
[sierramadre] cached: 18091 pairs (532 touching)
[sierranevadaspain] cached: 603 pairs (46 touching)
[vallefertil] cached: 808 pairs (27 touching)
[taiwan] cached: 35873 pairs (6082 touching)
[toano] cached: 182 pairs (65 touching)
[troodos] cached: 1169 pairs (332 touching)
[tsugaru] cached: 54 pairs (23 touching)
[yoro] cached: 31 pairs (11 touching)

Per-basin loop done.


In [5]:
# ─── Per-basin stats table ────────────────────────────────────────────
stats_df = pd.DataFrame(basin_stats)
stats_csv = RESULTS_DIR / f"build_earth_features_{regime.name}_stats.csv"
stats_df.to_csv(stats_csv, index=False)
print(f"Stats written → {stats_csv}")
stats_df

Stats written → /Users/guypi/Projects/channel-heads/data/results/build_earth_features_regC_stats.csv


,basin,from_cache,n_pairs,n_touching,n_not_touching,time_s,error
0,calnalpine,True,78,7,71,0.0,None
1,daqing,True,92,8,84,0.0,None
2,finisterre,True,15776,475,15301,0.0,None
3,humboldt,True,144,13,131,0.0,None
4,inyo,True,178,26,152,0.0,None
5,kammanasie,True,213,16,197,0.0,None
6,luliang,True,544,25,519,0.0,None
7,panamint,True,732,46,686,0.0,None
8,sakhalin,True,444,30,414,0.0,None
9,sierramadre,True,18091,532,17559,0.0,None


## Step 2.5 — Assemble master dataset

Concat all surviving basins, drop trivial negatives via
`filter_hard_negatives`, subsample negatives to 3:1, write
`data/results/master_dataset_<regime>.csv`.

In [6]:
if not basin_records:
    raise RuntimeError("No basin records collected; check Step 2 output.")

df_combined = pd.concat(basin_records, ignore_index=True)
print(f"Combined: {len(df_combined):,} rows from {df_combined['basin'].nunique()} basins")
print(f"  touching={int((df_combined['y']==1).sum())}  not={int((df_combined['y']==0).sum())}")

df_filtered = filter_hard_negatives(
    df_combined,
    max_L_ratio=HARD_NEG_MAX_L_RATIO,
    max_dist_ratio=HARD_NEG_MAX_DIST_RATIO,
)
print(f"\nAfter filter_hard_negatives: {len(df_filtered):,} rows")
print(f"  touching={int((df_filtered['y']==1).sum())}  not={int((df_filtered['y']==0).sum())}")

df_master = stratified_subsample_negatives(
    df_filtered, target_ratio=NEGATIVE_RATIO, random_state=RANDOM_SEED
)
print(f"\nAfter subsample (target neg:pos = {NEGATIVE_RATIO}:1): {len(df_master):,} rows")
print(f"  touching={int((df_master['y']==1).sum())}  not={int((df_master['y']==0).sum())}")

master_csv = RESULTS_DIR / f"master_dataset_{regime.name}.csv"
df_master.to_csv(master_csv, index=False)
print(f"\nWrote master dataset → {master_csv}")
df_master.head()

Combined: 75,012 rows from 17 basins
  touching=7764  not=67248

After filter_hard_negatives: 42,708 rows
  touching=7764  not=34944

After subsample (target neg:pos = 3.0:1): 31,056 rows
  touching=7764  not=23292

Wrote master dataset → /Users/guypi/Projects/channel-heads/data/results/master_dataset_regC.csv


,outlet,confluence,head_1,head_2,touching,contact_px,size1_px,size2_px,skipped_prefilter,y,...,orientation_diff_deg,headhead_dist_m,headhead_dist_norm,apex_angle_deg,strahler_order_diff,proximity_mean_m,proximity_max_m,proximity_profile_norm,qc_flags,basin
0,233,134,41,55,False,0,NaN,NaN,True,0,...,34.200920,1090.967454,0.252013,26.123397,0.0,637.480249,1090.967454,0.584326,NaN,calnalpine
1,233,134,41,92,False,0,NaN,NaN,True,0,...,77.566286,1849.158675,0.442303,51.546291,0.0,858.067672,1849.158675,0.464031,NaN,calnalpine
2,233,197,92,146,False,0,NaN,NaN,True,0,...,0.000000,1057.296922,0.195987,21.929588,1.0,827.344523,1083.060128,0.763895,NaN,calnalpine
3,233,199,146,265,False,0,NaN,NaN,True,0,...,52.790895,2271.994814,0.509962,68.010512,1.0,1296.148618,2271.994814,0.570489,NaN,calnalpine
4,233,199,228,265,False,0,NaN,NaN,True,0,...,17.857181,1161.046030,0.264593,32.933129,1.0,695.210687,1161.046030,0.598780,NaN,calnalpine


## Step 3 — CNN patches under the regime-pruned StreamObjects

Shells out to `scripts/build_cnn_patches_regime.py`. Patches go to
`data/results/_rasters_<regime>/{basin}/rasters/{outlet}_{h1}_{h2}.npy` and a
`data/results/raster_manifest_<regime>.csv` lists them.

In [ ]:
def run_script(args: list[str]) -> int:
    """Run a child Python script with live stdout/stderr (no buffering games)."""
    print("$ python " + " ".join(args))
    proc = subprocess.Popen(
        [sys.executable] + args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        cwd=str(PROJECT_ROOT),
    )
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

rc = run_script([
    "scripts/build_cnn_patches_regime.py",
    "--regime", regime.name,
    "-v",
])
print(f"\nStep 3 exit code: {rc}")
assert rc == 0, "Step 3 failed; check log above."

$ python scripts/build_cnn_patches_regime.py --regime regC -v
09:09:34 INFO build_cnn_patches_regime: Regime regC: T=0.100 km^2, pre_remove<=1, order_gap>=4
09:09:34 INFO build_cnn_patches_regime: Rasters output root: /Users/guypi/Projects/channel-heads/data/results/_rasters_regC


## Step 4 — Train CNN on the regime patches

Shells out to `scripts/train_cnn_regime.py`. Saves
`models/cnn_outlet_<regime>.pt` + training history CSV.

In [ ]:
rc = run_script([
    "scripts/train_cnn_regime.py",
    "--regime", regime.name,
    "-v",
])
print(f"\nStep 4 exit code: {rc}")
assert rc == 0, "Step 4 failed; check log above."

## Step 5 — Train combined XGBoost (geom + 4 CNN embeddings)

Shells out to `scripts/train_combined_xgb_regime.py`. Saves
`models/xgb_geom_plus_cnn_emb_<regime>.json` + feature columns + threshold.

In [ ]:
rc = run_script([
    "scripts/train_combined_xgb_regime.py",
    "--regime", regime.name,
    "-v",
])
print(f"\nStep 5 exit code: {rc}")
assert rc == 0, "Step 5 failed; check log above."

## Step 6 — Mars inference under the regime models

Shells out to `scripts/run_mars_combined_regime.py`. Reuses existing Mars
Phase 1-5 artifacts (vector network, patches, 5-feat table). Writes
`mars_combined_<regime>_predictions.{parquet,csv,gpkg}` +
`mars_combined_<regime>_by_network.csv`.

In [ ]:
rc = run_script([
    "scripts/run_mars_combined_regime.py",
    "--regime", regime.name,
    "-v",
])
print(f"\nStep 6 exit code: {rc}")
assert rc == 0, "Step 6 failed; check log above."

## Compare regime predictions vs Phase 6C baseline

Loads the regime's Mars predictions and the Phase 6C `emb` baseline
(production CNN + production XGB combined). Reports agreement, prediction
counts, and per-network shifts.

In [ ]:
regime_pq = PROJECT_ROOT / f"data/Mars/model_outputs/mars_combined_{regime.name}_predictions.parquet"
baseline_pq = PROJECT_ROOT / "data/Mars/model_outputs/mars_combined_model_predictions.parquet"

df_regime = pd.read_parquet(regime_pq)
df_base = pd.read_parquet(baseline_pq)

# Phase 6C parquet carries two combined variants (emb + logit). Use emb columns.
baseline_pred_col = "pred_touching_emb" if "pred_touching_emb" in df_base.columns else "pred_touching"
baseline_prob_col = "prob_touching_emb" if "prob_touching_emb" in df_base.columns else "prob_touching"

join = df_regime[["pair_id", "prob_touching", "pred_touching"]].merge(
    df_base[["pair_id", baseline_prob_col, baseline_pred_col]].rename(
        columns={baseline_prob_col: "base_prob", baseline_pred_col: "base_pred"}
    ),
    on="pair_id", how="inner",
)
agree = int((join["pred_touching"] == join["base_pred"]).sum())
n = len(join)
print(f"Regime ({regime.name}) vs Phase 6C baseline (emb):")
print(f"  pairs joined           : {n}")
print(f"  regime touching        : {int(df_regime['pred_touching'].sum())} / {len(df_regime)}")
print(f"  baseline touching      : {int(df_base[baseline_pred_col].sum())} / {len(df_base)}")
print(f"  agreement              : {agree}/{n} = {100.0*agree/n:.1f} %")
join.head()

In [ ]:
# Probability shift histogram: regime - baseline.
import matplotlib.pyplot as plt

diff = join["prob_touching"] - join["base_prob"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(diff, bins=40, color="#1f6fb4", alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel(f"prob({regime.name}) − prob(phase6c emb)")
ax.set_ylabel("# Mars pairs")
ax.set_title(
    f"Mars touching-probability shift under {regime.name}\n"
    f"(median = {diff.median():+.3f}, mean = {diff.mean():+.3f})"
)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()